In [ ]:
# ---------------------------------------------------------------------------
# Background overlap sensitivity analysis
# ---------------------------------------------------------------------------
# This script quantifies how much background signal contaminates the
# gray-value ranges occupied by root and sand voxels across multiple scan
# sessions. For each material class (root, sand), the script identifies the
# bins that together account for a given fraction of the class intensity
# distribution (the "top-mass" set), then measures what percentage of the
# background distribution falls within that same intensity range. This is
# repeated across coverage fractions from 10 % to 90 % to produce a
# sensitivity curve: a steep rise indicates that background and the material
# class strongly overlap in gray value, making clean segmentation harder.
# Results are averaged across all scan sessions and displayed as a dual-axis
# line plot with ± SD bands, saved as a vector PDF for publication.
# ---------------------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Configuration — replace these values before running
# ---------------------------------------------------------------------------
# Absolute paths to the per-class histogram CSVs for the session to analyse.
# Each CSV must contain two columns: gray-value intensity and pixel count,
# with no header row.
# To switch sessions, update these three paths accordingly.
root_csv = "/path/to/root_histogram.csv"
sand_csv = "/path/to/sand_histogram.csv"
bg_csv   = "/path/to/background_histogram.csv"

# Absolute path (including filename) for the saved output figure.
output_figure = "/path/to/output/background_overlap_dual_axis.pdf"

# Coverage fraction used for the single-session diagnostic print.
# Must be one of the values in the `coverage` list below (default: 0.9).
mass = 0.9

# ---------------------------------------------------------------------------
# Load per-class histograms
# ---------------------------------------------------------------------------
root = pd.read_csv(root_csv, header=None, names=["intensity", "count"])
sand = pd.read_csv(sand_csv, header=None, names=["intensity", "count"])
bg   = pd.read_csv(bg_csv,   header=None, names=["intensity", "count"])

# ---------------------------------------------------------------------------
# Merge histograms into a single aligned dataframe
# ---------------------------------------------------------------------------
# An outer join on intensity ensures that bins present in one class but not
# another are retained, with missing counts filled as zero.
df = (bg.rename(columns={"count": "bg"})
        .merge(root.rename(columns={"count": "root"}), on="intensity", how="outer")
        .merge(sand.rename(columns={"count": "sand"}), on="intensity", how="outer")
        .fillna(0)
        .sort_values("intensity"))

# ---------------------------------------------------------------------------
# Convert raw counts to probability mass functions (PMFs)
# ---------------------------------------------------------------------------
# Normalizing by the total count of each class converts bin counts into
# probabilities, allowing fair comparison across classes with different
# total voxel counts.
for col in ["bg", "root", "sand"]:
    total = df[col].sum()
    if total == 0:
        raise ValueError(f"No counts found for class '{col}'. Check the input CSV.")
    df[col + "_p"] = df[col] / total

# ---------------------------------------------------------------------------
# Top-mass mask helper function
# ---------------------------------------------------------------------------
def top_mass_mask(p_series, mass=0.80):
    """Return a boolean mask selecting the smallest set of intensity bins
    whose combined probability mass equals or exceeds `mass`, selecting
    the highest-probability bins first (analogous to a highest posterior
    density interval for discrete distributions).

    Parameters
    ----------
    p_series : pd.Series
        PMF values indexed by intensity bin.
    mass : float
        Target cumulative probability mass (e.g. 0.80 for 80 %).

    Returns
    -------
    pd.Series of bool
        True for bins included in the top-mass set.
    """
    order    = np.argsort(-p_series.values)          # descending by probability
    p_sorted = p_series.values[order]
    cum      = np.cumsum(p_sorted)
    k        = np.searchsorted(cum, mass) + 1        # minimum bins needed to reach `mass`
    idx_keep = set(p_series.index[order[:k]])
    return p_series.index.to_series().isin(idx_keep)

# ---------------------------------------------------------------------------
# Single-session diagnostic — background contamination at chosen mass level
# ---------------------------------------------------------------------------
root_core = top_mass_mask(df["root_p"], mass)
sand_core = top_mass_mask(df["sand_p"], mass)

bg_in_root_core = df.loc[root_core, "bg_p"].sum() * 100
bg_in_sand_core = df.loc[sand_core, "bg_p"].sum() * 100

print(f"Background within ROOT top-{int(mass*100)}% intensities: {bg_in_root_core:.2f}%")
print(f"Background within SAND top-{int(mass*100)}% intensities: {bg_in_sand_core:.2f}%")

# ---------------------------------------------------------------------------
# Cross-session sensitivity data
# ---------------------------------------------------------------------------
# Background overlap (%) at each coverage fraction for every scan session.
# Rows correspond to sessions; columns correspond to entries in `coverage`.
# Replace or extend these lists when adding new sessions.
coverage = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

bg_within_root_Col_S1 = [0.58, 1.17, 1.77, 2.45, 3.18, 4.11, 5.46,  7.61, 14.05]
bg_within_root_Col_S2 = [0.91, 1.81, 2.80, 3.89, 5.17, 6.74, 8.94, 12.47, 20.14]
bg_within_root_Mt_S1  = [0.70, 1.45, 2.30, 3.29, 4.58, 6.51,10.42, 18.22, 37.28]
bg_within_root_Mt_S2  = [0.79, 1.66, 2.77, 4.50, 7.86,14.65,27.23, 43.61, 62.76]
bg_within_root_Sav_S1 = [0.61, 1.22, 1.89, 2.66, 3.58, 4.88, 6.73, 10.01, 16.92]
bg_within_root_Sav_S2 = [0.93, 1.83, 2.64, 3.56, 4.77, 6.32, 8.82, 14.19, 31.01]

bg_within_sand_Col_S1 = [0.00, 0.00, 0.00, 0.00, 0.01, 0.01, 0.01, 0.01, 0.02]
bg_within_sand_Col_S2 = [0.00, 0.00, 0.01, 0.01, 0.01, 0.01, 0.02, 0.02, 0.03]
bg_within_sand_Mt_S1  = [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00]
bg_within_sand_Mt_S2  = [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.02]
bg_within_sand_Sav_S1 = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.63]
bg_within_sand_Sav_S2 = [0.00, 0.00, 0.00, 0.00, 0.00, 0.01, 0.01, 0.02, 0.10]

# ---------------------------------------------------------------------------
# Cross-session statistics
# ---------------------------------------------------------------------------
# Stack per-session arrays and compute the mean and SD across sessions at
# each coverage fraction. ddof=0 (population SD) is used here because all
# available sessions are included — this is not a sample from a larger pool.
root_arrays = [bg_within_root_Col_S1, bg_within_root_Col_S2, bg_within_root_Mt_S1,
               bg_within_root_Mt_S2,  bg_within_root_Sav_S1, bg_within_root_Sav_S2]
sand_arrays = [bg_within_sand_Col_S1, bg_within_sand_Col_S2, bg_within_sand_Mt_S1,
               bg_within_sand_Mt_S2,  bg_within_sand_Sav_S1, bg_within_sand_Sav_S2]

root_means = np.mean(root_arrays, axis=0)
root_stds  = np.std(root_arrays,  axis=0)
sand_means = np.mean(sand_arrays, axis=0)
sand_stds  = np.std(sand_arrays,  axis=0)

print(f"Root — mean overlap across sessions: {root_means}")
print(f"Root — SD   across sessions        : {root_stds}")
print(f"Sand — mean overlap across sessions: {sand_means}")
print(f"Sand — SD   across sessions        : {sand_stds}")

# ---------------------------------------------------------------------------
# Dual-axis sensitivity plot
# ---------------------------------------------------------------------------
# Root and sand are plotted on separate y-axes because their overlap ranges
# differ by roughly two orders of magnitude; a shared axis would compress
# the sand curve into an unreadable flat line.
fig, ax1 = plt.subplots(figsize=(5, 4))

color_root = 'cornflowerblue'
color_sand = 'gold'

# Primary y-axis — root overlap
ax1.set_xlabel("Fraction of top-occurring intensities considered")
ax1.set_ylabel("Root overlap (%)", color=color_root)
ax1.set_ylim(-0.25, 50.25)
ax1.plot(coverage, root_means, marker='o', color=color_root,
         linewidth=2.5, label="Root mean", zorder=5)
ax1.fill_between(coverage,
                 root_means - root_stds,
                 root_means + root_stds,
                 color=color_root, alpha=0.2, label="Root ± SD")
ax1.tick_params(axis='y', labelcolor='black')

# Secondary y-axis — sand overlap (zoomed to its much smaller range)
ax2 = ax1.twinx()
ax2.set_ylabel("Sand overlap (%)", color=color_sand)
ax2.set_ylim(-0.0025, 0.4)
ax2.plot(coverage, sand_means, marker='s', color=color_sand,
         linewidth=2.5, label="Sand mean", zorder=5)
ax2.fill_between(coverage,
                 sand_means - sand_stds,
                 sand_means + sand_stds,
                 color=color_sand, alpha=0.2, label="Sand ± SD")
ax2.tick_params(axis='y', labelcolor='black')

fig.legend(loc="upper left", bbox_to_anchor=(0.15, 0.95), frameon=False)
fig.tight_layout()
plt.savefig(output_figure, dpi=300, bbox_inches="tight")
plt.show()